# AGENT10 (en construction)

La structure des transformations possibles de l'environnement par rapport à l'agent peut prendre virtuellement une infinité de formes différentes. 

Pour parvenir à capturer cette structure de transformation, l'agent doit implémenter des présuppositions. 

Il nous faut un moyen de représenter ces présuppositions qui prend en compte:

* Le modèle corporel de l'agent: la position des interactions par rapport à son propre corps.
* Le modèle de déplacement : Les transformations spatiales liées aux interactions associées aux déplacements de l'agent
* Le modèle d'objets : comment différentes interactions peuvent etre considérées _afforded_ par un même objet

De plus, l'agent a besoin de principes motivationnels qui l'orientent à travers ce processus d'apprentissage.

Par exemple, faire des experimentations qui permettent d'isoler des causes (construire une model causal)


L'agent 10 réimplémente l'agent 8 avec les principes tirés de l'agent 9: 

* Les positions des interactions sont stockées dans le `body_model`.
* Les transformations spatiales sont stockées dans le `displacement_model` qui pourrait aussi s'appeler `action_model`.
* Les règles de création et suppression d'objets sont stockées dans le `object model`.

Il y a deux mémoires spatiales

* La mémoire égocentrée des interactions
* La mémoire des instances d'objets (chacune instance d'objet localisé renvoie à un `object model` pour ce type d'objet)

Travaux en relation: 

On peut faire le lien avec un problème identifié par Hod Lipson et ses coauteurs : permettre à un robot de reconfigurer son self-model en cas de modification physique. 
[Resilient Machines Through Continuous Self-Modeling](https://www.science.org/doi/full/10.1126/science.1133687) Josh Bongard, Victor Zykov, and Hod Lipson (2006)

# Généralisation des schémas spatio séquentiels

D'une manière générale, il faut pouvoir représenter des motifs spatio sequentiels d'événements. 

Changer les interactions composites pour qu'elles stockent leurs sous-interactions dans un dictionnaire dont la clé représente les coordonnées spatio-temporelles, et dont la valeur contient la sous interaction. 

* Ré-écrire les schémas en stockant les pre-interaction et post-interaction dans un dictionnaire en clé 0 et 1.
* Permettre des interactions composites spatiales : leurs sous interaction ont des positions spatiales différentes mais ont toutes une coordonnée temporelle de 0.  

Il faut pouvoir appliquer des déplacements spatiaux sur les schémas (rotation et translation)

# Généralisation du mécanisme de construction de schémas

Le contexte doit pouvoir contenir plusieurs interactions enactées au meme moment à des positions différentes

* Quand deux interactions sont enactées au même moment on crée ou renforce une interaction composite spatiale qui les associe

Problème: comment attribue-t-on une position à une interaction enactée ?


# Experimentation minimaliste

## Piste d'une seule décision pouvant générer plusieurs événements d'interaction

La décision `feel` peut générer trois événements d'interaction placés à des endroits différents initialement arbitraires. 

* on veut que l'agent apprenne à placer les trois événements feel d'une façon cohérente avec da capacité de transformation spatiale


## Piste des décisions parallèles 

L'agent peut enacter plusieurs interactions feel simultanées. Au départ il leur attribue une position arbitraire. On veut qu'il apprenne à leur attribuer une position bien ordonnée. 

* Il faut pouvoir prendre plusieurs (deux) décisions simultanées chacune attribuée à une position différentes dans le `body model`.
* il faut un contexte qui enregistre les interactions énactées simultanées

Au départ les deux décisions sont indépendantes mais on veut voir apparaitre des décisions qui composites qui sélectionnent les deux décisions primitives ensembles



# L'environnement Small Loop

Nous repartons de notesbooks/Agent14 et Modifions pour renvoyer une liste d'événements d'interaction:

* La décision `FEEL` renvoie une liste de trois trois outcomes `WALL` ou `EMPTY`. La position dans la liste est liée à la position par rapport à l'agent.
* On supprime les décisions `FEEL_LEFT` et `FEEL_FRONT`
* Les autres décisions renvoient une liste contenant un seul événement d'interaction comme avant.
* L'environnement est rendu torique grace à `np.mod(position, ENV_SIZE)`

In [13]:
# save_dir = "sav"

FORWARD = 2  #0
FEEL = 1
# FEEL_LEFT = 2
# FEEL_RIGHT = 3
TURN_LEFT = 4
TURN_RIGHT = 5

# The size of the simulateor should be even to place the agent at the center
ENV_HIGHT = 5
ENV_WIDTH = 5
SIM_HIGHT = 5
SIM_WIDTH = 5

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from ipywidgets import Button, HBox,VBox, Output
from IPython.display import display

LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
EMPTY = 0
WALL = 1
FEELING_EMPTY = 2
FEELING_WALL = 3
BUMPING = 4
UNKNOWN = 5

colors = ["#b0b0b0", '#b0b0b0', '#ffffff', '#535865', "#F93943"]  # Hidden environment
colors = ["#D6D6D6", '#5C946E', '#FAE2DB', '#535865', "#F93943", 
          "#EEEEEE", "#85A48F", '#FAE2DB', '#535865', "#F93943", '#BAC9E1']  # Simulator
agent_color = "#1976D2"
simulator_agent_color = "#aaaaaa"
prediction_error_color = "#f62dae"
agent_size = 200
ENV_SIZE = np.array([ENV_HIGHT, ENV_WIDTH])
DIRECTIONS = np.array([
            [0, -1],  # Left
            [1, 0],   # Down
            [0, 1],   # Right
            [-1, 0]   # Up
            ])

class SmallLoop():
    def __init__(self, position, direction, grid):
        self.environment_grid = np.array(grid)
        self.display_grid = np.full((ENV_HIGHT, ENV_WIDTH + SIM_WIDTH), WALL, dtype=int)
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid
        self.position = np.array(position) 
        self.direction = direction
        self.cmap = ListedColormap(colors)
        self.norm = BoundaryNorm(np.arange(-0.5, len(colors) + 0.5, 1.0), self.cmap.N)
        self.marker_size = agent_size
        self.marker_map = {LEFT: '<', DOWN: 'v', RIGHT: '>', UP: '^'}
        self.marker_color = agent_color

    def outcome(self, action):
        """Update the grid. Return the outcome of the action."""
        result = [0]
        self.display_grid[0:self.environment_grid.shape[0], 0:self.environment_grid.shape[1]] = self.environment_grid

        if action == FORWARD:  
            target_position = np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE)
            if self.environment_grid[tuple(target_position)] == EMPTY:
                self.position[:] = target_position
            else:
                result = [1]
                self.display_grid[tuple(target_position)] = BUMPING
        
        elif action == TURN_RIGHT:
            self.direction = {LEFT: UP, DOWN: LEFT, RIGHT: DOWN, UP: RIGHT}[self.direction]
        
        elif action == TURN_LEFT:
            self.direction = {LEFT: DOWN, DOWN: RIGHT, RIGHT: UP, UP: LEFT}[self.direction]
        
        elif action == FEEL:
            # Decision FEEL_FRONT now yields 3 outcomes for front, left, and Right
            result = []
            for feeling_position in [np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE), # Front
                                     np.mod(self.position + DIRECTIONS[(self.direction + 1) % 4], ENV_SIZE), # Left
                                     np.mod(self.position + DIRECTIONS[self.direction - 1], ENV_SIZE)]: # Right
                if self.environment_grid[tuple(feeling_position)] == EMPTY:
                    self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
                    result.append(0)
                else:
                    result.append(1)
                    self.display_grid[tuple(feeling_position)] = FEELING_WALL

        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
    
    def display(self, simulator=None):
        """Display the grid in the notebook"""
        out.clear_output(wait=True)
        with out:
            fig, ax = plt.subplots()
            if simulator is not None:
                plt.scatter(simulator.position[1] + ENV_WIDTH, simulator.position[0], s=self.marker_size, marker=self.marker_map[simulator.direction], 
                            c=simulator.marker_color)
                self.display_grid[0:SIM_HIGHT, ENV_WIDTH:(ENV_WIDTH + SIM_WIDTH + 1)] = simulator.display_grid[0:SIM_HIGHT, 0:SIM_WIDTH] + 5
            ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            ax.text(4.5, 0, f"{step:>3}", fontsize=12, color='White')
            plt.show()
            plt.pause(0.01)
    
    def save(self, step, img_nb, simulator):
        """Save the display as a PNG file"""
        fig, ax = plt.subplots()
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis('off')
        ax.imshow(self.display_grid, cmap=self.cmap, norm=self.norm)
        plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
        plt.scatter(simulator.position[1] + 6, simulator.position[0], s=simulator.marker_size, marker=self.marker_map[UP], 
                    c=simulator.marker_color)
        ax.text(4.5, 0, f"{step:>4}", fontsize=10, color='White')
        plt.savefig(f"{save_dir}/{img_nb:04}.png", bbox_inches='tight', pad_inches=0, transparent=True)
        plt.close(fig)
    
    def clear(self, clear):
        """Clear the grid display"""
        if clear:
            self.display_grid[0:6, 0:6] = self.environment_grid
       

# Les classes Interaction

Nous repartons de l'agent notebooks/Agent11 avec les modificaitons: 
* Les interactions composites sont construites à partir d'un dictionnaire d'interactions
* La méthode séquence renvoie la liste des interactions primitives
* Nous créons la classe LocalizedInteraction

In [15]:
class CompositeInteraction:
    """A composite interaction is a tuple (pre_interaction, post_interaction) and a weight"""
    def __init__(self, interactions):
        # Interactions have coordinates: 2D space and 1D time
        # self.interactions = {(0, 0, 0):pre_interaction, (0, 0, 1):post_interaction}
        self.interactions = interactions
        # self.pre_interaction = pre_interaction
        # self.post_interaction = post_interaction
        self.weight = 1
        self._step = 1

    def get_pre_interaction(self):
        # Prepare for composite interaction that may contain several interactions at different position at time 0
        pre_interactions = [p for k, p in self.interactions.items() if k[1]==0]
        return pre_interactions[0]
        # return self.interactions[0]

    def get_post_interaction(self):
        # Prepare for composite interactions that may contain several interactions at different positions at time 1
        post_interactions = [p for k, p in self.interactions.items() if k[1]==1]
        return post_interactions[0]
        # return self.interactions[1]
        
    def get_decision(self):
        """Return the flatten sequence of intermediary primitive interactions terminated with the final decision"""
        return f"{self.get_pre_interaction().sequence()}{self.get_post_interaction().get_decision()}"

    def get_actions(self):
        """Return the flat sequence of the decisions of this interaction as a string"""
        return f"{self.get_pre_interaction().get_actions()}{self.get_post_interaction().get_actions()}"
    
    def get_valence(self):
        """Return the valence of the pre_interaction plus the valence of the post_interaction"""
        return self.get_pre_interaction().get_valence() + self.get_post_interaction().get_valence()

    def reinforce(self):
        """Increment the composite interaction's weight"""
        self.weight += 1

    def key(self):
        """ The key to find this interaction in the dictionary is the string '<pre_interaction><post_interaction>'. """
        return f"({self.get_pre_interaction().key()},{self.get_post_interaction().key()})"

    def pre_key(self):
        """Return the key of the pre_interaction"""
        #if self.weight > confidence_threshold:
        return self.get_pre_interaction().key()
        #else:
        #return self.pre_interaction.pre_key()

    def __str__(self):
        """ Print the interaction in the Newick tree format (pre_interaction, post_interaction: valence) """
        return f"({self.get_pre_interaction()}, {self.get_post_interaction()}: {self.weight})"

    def __eq__(self, other):
        """ Interactions are equal if they have the same pre and post interactions """
        if isinstance(other, self.__class__):
            return (self.get_pre_interaction() == other.get_pre_interaction()) and (self.get_post_interaction() == other.get_post_interaction())
        else:
            return False

    def get_length(self):
        """Return the length of the number of primitive interactions in this composite interaction"""
        return self.get_pre_interaction().get_length() + self.get_post_interaction().get_length()

    def increment(self, interaction, interactions):
        """Increment the step of the appropriate sub-interaction. Return the enacted interaction if it is over, or None if it is ongoing."""
        # First step 
        if self._step == 1:
            interaction = self.get_pre_interaction().increment(interaction, interactions)
            # Ongoing pre-interaction. Return None
            if interaction is None:
                return None
            # Pre-interaction succeeded. Increment the step and return None
            elif interaction == self.get_pre_interaction():
                self._step = 2
                return None
            # Pre-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                return interaction
        # Second step
        else:
            interaction = self.get_post_interaction().increment(interaction, interactions)
            # Ongoing post-interaction. Return None
            if interaction is None:
                return None
            # Post-interaction succeeded. Reset the step and return this interaction
            elif interaction == self.get_post_interaction():
                self._step = 1
                return self
            # Post-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                interactions = {(0, 0):self.get_pre_interaction(), (0, 1):interaction}
                # composite_interaction = CompositeInteraction(self.get_pre_interaction(), interaction)
                composite_interaction = CompositeInteraction(interactions)
                if composite_interaction.key() not in interactions:
                    # Add the enacted composite interaction to memory
                    interactions[composite_interaction.key()] = composite_interaction
                    # print(f"Learning {composite_interaction}")
                    return composite_interaction
                else:
                    # Reinforce the existing composite interaction and return it
                    interactions[composite_interaction.key()].reinforce()
                    # print(f"Reinforcing {interactions[composite_interaction.key()]}")
                    return interactions[composite_interaction.key()]

    def current(self):
        """Return the current intended primitive interaction"""
        # Step 1: the current primitive interaction of the pre-interaction
        if self._step == 1:
            return self.get_pre_interaction().current()
        # Step 2: The current primitive interaction of the post-interaction
        else:
            return self.get_post_interaction().current()

    def sequence(self):
        """Return the flat list of primitive interactions of this composite interaction"""
        return self.get_pre_interaction().sequence() + self.get_post_interaction().sequence()
        # return f"{self.get_pre_interaction().sequence()}{self.get_post_interaction().sequence()}"

    def get_post_interactions(self):
        """Return the list of the hierarchy of the sub post_interactions"""
        return [self.get_post_interaction()] + self.get_post_interaction().get_post_interactions()

In [16]:
class Interaction:
    """An interaction is a tuple (action, outcome) with a valence"""
    def __init__(self, _action, _outcome, _valence):
        self._action = _action
        self._outcome = _outcome
        self._valence = _valence
        self.weight = 10
        
    def get_action(self):
        """Return the action"""
        return self._action

    def get_actions(self):
        """Return the action as a string for compatibilty with CompositeInteraction"""
        return str(self._action)

    def get_decision(self):
        """Return the decision key"""
        return f"{self._action}"
        # return f"a{self._action}"

    #def get_primitive_action(self):
    #    """Return the action for compatibility with CompositeInteraction"""
    #    return self._action

    def get_outcome(self):
        """Return the action"""
        return self._outcome

    def get_valence(self):
        """Return the action"""
        return self._valence

    def key(self):
        """ The key to find this interaction in the dictinary is the string '<action><outcome>'. """
        return f"{self._action}{self._outcome}"

    def pre_key(self):
        """Return the key. Used for compatibility with CompositeInteraction"""
        return ""  # self.key()

    def __str__(self):
        """ Print interaction in the form '<action><outcome:<valence>' for debug."""
        return f"{self._action}{self._outcome}:{self._valence}"

    def __eq__(self, other):
        """ Interactions are equal if they have the same key """
        if isinstance(other, self.__class__):
            return self.key() == other.key()
        else:
            return False

    def get_length(self):
        """The length of the sequence of this interaction"""
        return 1

    def increment(self, interaction, interactions):
        """Return the enacted interaction for compatibility with composite interactions"""
        return interaction

    def current(self):
        """Return itself for compatibility with composite interactions"""
        return self

    def sequence(self):
        """Return the key. Use for compatibility with composite interactions"""
        return [self]
        # return self.key()

    def get_post_interactions(self):
        """Return the empty list for compatibility with composite interactions"""
        return []

## Classe LocalizedInteraction

Les LocalizedInteractions sont utilisés pour synchroniser le simulateur.

In [17]:
class LocalizedInteraction:
    """A LocalizedInteraction is an interaction associated with a 2D spatial position"""
    def __init__(self, _interaction, _position):
        self.interaction = _interaction
        self.position = _position
    
    def __repr__(self):
        """Print the interaction and its position"""
        return f"{self.interaction}at{self.position}"

    def key(self):
        """The key is a tuble made of the 2D position and the interaction key"""
        return (self.position, self.interaction.key())
        # 2D position
        # return (self.position[0], self.position[1], self.interaction.key())

    def __eq__(self):
        """Localized interactions are equal if they have the same keys: same position and same interaction"""
        if isinstance(other, self.__class__):
            return self.key() == other.key()
        else:
            return False

# L'agent

Nous repartons de l'Agent11. 

Il est modifié pour: 
* Traiter une liste d'outcomes reçus de l'environnement
* Maintenir un contexte qui peut contenir plusieurs dévénemtents d'interaction enactés
    * (En construction: pour l'instant il ne tient compte que de `outcome[0]`)

In [27]:
import pandas as pd
trace = True

class Agent:
    def __init__(self, _interactions, simulator):
        """ Initialize our agent """
        self._interactions = {interaction.key(): interaction for interaction in _interactions}
        self._primitive_intended_interaction = self._interactions[f"{FORWARD}0"]
        self._intended_interaction = None
        self._simulator = simulator
        # The context
        self._penultimate_interaction = None
        self._previous_interaction = None
        self._last_interaction = None
        self._penultimate_composite_interaction = None
        self._previous_composite_interaction = None
        self._last_composite_interaction = None
        self._permutation_index = 0
        # self._simulated_outcomes = []
        
        # Prepare the dataframe of proposed interactions
        default_interactions = [interaction for interaction in _interactions if interaction.get_outcome() == 0]
        data = {'activated': [""] * len(default_interactions),
                'weight': [0] * len(default_interactions),
                'actions': [i.get_actions() for i in default_interactions],
                'intention': [i.key() for i in default_interactions],
                'valence': [i.get_valence() for i in default_interactions],
                'decision': [i.get_decision() for i in default_interactions],
                'length': [1] * len(default_interactions),
                'pre': [""] * len(default_interactions)} 
        self._default_df = pd.DataFrame(data)
        self.proposed_df = None
        self.decision_df = None
        self.clear = True # Used to clear the display after the enacted interaction

    def action(self, _outcome):
        """Implement the agent's policy"""

        # The list of enacted primitive interactions
        primitive_enacted_interactions = [self._interactions[f"{self._primitive_intended_interaction.get_action()}{o}"] for o in _outcome]
        # The list of localized enacted primitive interactions based on the rank of the outcome
        localized_interactions = [LocalizedInteraction(interaction, rank) for rank, interaction in enumerate(primitive_enacted_interactions)]

        # Trace the previous cycle
        if trace:
            print(
            f"Enacted primitive localized interactions: {localized_interactions},\n" 
            f"Sequential predicted outcome: {self._primitive_intended_interaction.get_outcome()}, "
            # f"Simulator predicted outcomes: {self._simulated_outcomes}, "
            f"Prediction_correct: {self._primitive_intended_interaction.get_outcome() == _outcome[0]}, ")

        # Synchronize the simulator
        consistent = self._simulator.synchronize(localized_interactions)
        if not consistent:
            print("-- Simulator inconsistent. GEITH models need reajusting!")
            self._simulator.reset()
            self.permutate_models()

        # For now, only use the first primitive enacted interaction
        primitive_enacted_interaction = primitive_enacted_interactions[0]
        
        # Follow up the enaction
        if self._intended_interaction is None: # First interaction cycle
            enacted_interaction = primitive_enacted_interaction
        else:
            enacted_interaction = self._intended_interaction.increment(primitive_enacted_interaction, self._interactions)

        # If the intended interaction is over (completely enacted or aborted)
        if enacted_interaction is None:
            self.clear = False
        else:
            self.clear = True
            # Memorize the context
            self._penultimate_composite_interaction = self._previous_composite_interaction
            self._previous_composite_interaction = self._last_composite_interaction
            self._penultimate_interaction = self._previous_interaction
            self._previous_interaction = self._last_interaction
            self._last_interaction = enacted_interaction
            # Call the learning mechanism
            self.learn(enacted_interaction)
            # Create the proposed dataframe
            self.create_proposed_df()
            self.aggregate_propositions()
            # Decide the next enaction
            self.decide()
            # Simulate the intended interaction in a clone of the simulator 
            # self.simulate()

        # Return the next primitive action
        self._primitive_intended_interaction = self._intended_interaction.current()        
        return self._primitive_intended_interaction.get_action()
        
    def learn(self, enacted_interaction):
        """Learn the composite interactions"""
        # First level of composite interactions
        self._last_composite_interaction = self.learn_composite_interaction(self._previous_interaction, enacted_interaction)
        # Second level of composite interactions
        self.learn_composite_interaction(self._previous_composite_interaction, enacted_interaction)
        self.learn_composite_interaction(self._penultimate_interaction, self._last_composite_interaction)

        # Higher level composite interaction made of two composite interactions
        if self._last_composite_interaction is not None:
            self.learn_composite_interaction(self._penultimate_composite_interaction, self._last_composite_interaction)

    def learn_composite_interaction(self, pre_interaction, post_interaction):
        """Record or reinforce the composite interaction made of (pre_interaction, post_interaction)"""
        if pre_interaction is None:
            return None
        else:
            # If the pre-interaction exists
            interactions = {(0, 0):pre_interaction, (0, 1):post_interaction}
            # composite_interaction = CompositeInteraction(pre_interaction, post_interaction)
            composite_interaction = CompositeInteraction(interactions)
            if composite_interaction.key() not in self._interactions:
                # Add the composite interaction to memory
                self._interactions[composite_interaction.key()] = composite_interaction
                if trace:
                    print(f"Learning {composite_interaction}")
                return composite_interaction
            else:
                # Reinforce the existing composite interaction and return it
                self._interactions[composite_interaction.key()].reinforce()
                if trace:
                    print(f"Reinforcing {self._interactions[composite_interaction.key()]}")
                return self._interactions[composite_interaction.key()]

    def create_proposed_df(self):
        """Create the proposed dataframe from the activated interactions"""
        # The list of activated interactions that match the current context
        activated_interactions = [i for i in self._interactions.values() if i.get_length() > 1 
                                  and i.get_pre_interaction() in self._last_composite_interaction.get_post_interactions()]
        data = {'activated': [i.key() for i in activated_interactions],
                'weight': [i.weight for i in activated_interactions],
                'actions': [i.get_post_interaction().get_actions() for i in activated_interactions],
                'intention': [i.get_post_interaction().key() for i in activated_interactions],
                'valence': [i.get_post_interaction().get_valence() for i in activated_interactions],
                'decision': [i.get_post_interaction().get_decision() for i in activated_interactions],
                'pre': [i.get_post_interaction().pre_key() for i in activated_interactions],
                'length': [i.get_post_interaction().get_length() for i in activated_interactions],
                }
        activated_df = pd.DataFrame(data).astype(self._default_df.dtypes)  # Force the same types in case activated_df is empty

        # Create the proposed dataframe
        self.proposed_df = pd.concat([self._default_df, activated_df], ignore_index=True).sort_values(by='decision', ascending=True).reset_index(drop=True)

        # Calculate the proclivity of each proposition
        self.proposed_df['proclivity'] = self.proposed_df['weight'] * self.proposed_df['valence']

        # Compute the probability of each propositions
        # self.proposed_df['probability'] = self.proposed_df['weight'] / self.proposed_df.groupby('actions')['weight'].transform('sum')
        self.proposed_df['probability'] = self.proposed_df.groupby('intention')['weight'].transform('sum') / self.proposed_df.groupby('actions')['weight'].transform('sum')

    def aggregate_propositions(self):
        """Aggregate the proclivity"""
        # Aggregate the proclivity for each decision
        grouped_df = self.proposed_df.groupby('decision').agg({'proclivity': 'sum', 'actions': 'first', # 'action': 'first', 
                                                               'length': 'first', 'intention': 'first', 'pre': 'first'}).reset_index()
        # For each proposed composite decision 
        for index, proposed in grouped_df[grouped_df['length'] > 1].iterrows():
            # print(f"Index {index}, actions {proposition['actions']}, intention {proposition['intention']}")
            # Find shorter decisions that start with the same sequence 
            for _, shorter in self.proposed_df[self.proposed_df.apply(lambda row: proposed['actions'].startswith(row['actions']) 
                                                                      and row['length'] < proposed['length'], axis=1)].iterrows():
                # Add the proclivity of the shorter decisions
                grouped_df.loc[index, 'proclivity'] += shorter['proclivity']
                # print(f"Decision {proposed['decision']} recieves {shorter['proclivity']} from shorter {shorter['intention']}")
        
        # Sort by descending proclivity
        self.decision_df = grouped_df.sort_values(by=['proclivity', 'decision'], ascending=[False, True]).reset_index(drop=True)

    def decide(self):
        """Selects the intended_interaction at the top of the proposed dataframe"""
        # The intended interaction is in the first row because it has been sorted by descending proclivity
        intended_interaction_key = self.decision_df.loc[0, 'intention']
        if trace:
            print("Intention:", intended_interaction_key)
        self._intended_interaction = self._interactions[intended_interaction_key]

    def simulate(self):
        """Simulate the intended interaction in a cloned simulator and display it. """
        current_simulation = self._simulator.clone()
        for primitive_interaction in self._intended_interaction.sequence():
            primitive_decision = primitive_interaction.get_action()
            simulated_outcomes = current_simulation.predict(primitive_decision)
            consistent = simulated_outcomes[0] in [primitive_interaction.get_outcome(), UNKNOWN]
            print("Simulate decision:", primitive_decision, "anticipated outcomes:", simulated_outcomes, "consistent:", consistent)
            current_simulation.marker_color = simulator_agent_color if consistent else prediction_error_color
            e.display(current_simulation)

    def permutate_models(self):
        """"Try the next model in the list of permutations"""
        body_model[(FEEL, 0)] = model_permutations[self._permutation_index][0]
        body_model[(FEEL, 1)] = model_permutations[self._permutation_index][1]
        body_model[(FEEL, 2)] = model_permutations[self._permutation_index][2]
        action_model[f"{TURN_LEFT}0"] = [model_permutations[self._permutation_index][3]]
        action_model[f"{TURN_RIGHT}0"] = [model_permutations[self._permutation_index][4]]
        self._permutation_index = (self._permutation_index + 1) % 12

# Le simulateur

Nous reprenons le simulateur de geith/Agent5 avec les modifications: 
* Méthode `synchronize()` force l'état du simulateur sur la base des interactions primitives énactées
* Méthode `predict()` returns the predicted outcome of the ginven action

In [28]:
KILL = 0
SPAWN = 1
TRANSLATE = 2
ROTATE_COUNTERCLOCKWISE = 3
ROTATE_CLOCKWISE = 4

SIM_SIZE = np.array([SIM_HIGHT, SIM_WIDTH])

class Simulator():
    def __init__(self, position, direction, grid):
        self.init_grid = np.array(grid)
        self.simulation_grid = np.array(grid)
        self.display_grid = np.array(grid)
        self.position = np.array(position)  # Must clone the position
        self.marker_size = agent_size
        self.marker_color = simulator_agent_color
        self.direction = direction

    def translate(self):
        """Translate the grid when the agent moves forward"""
        self.position = np.mod(self.position + DIRECTIONS[self.direction], SIM_SIZE)
        # Egocentric simulation
        # self.simulation_grid[:SIM_HIGHT, :SIM_WIDTH] = np.vstack((np.full((1, self.simulation_grid.shape[1]), UNKNOWN), self.simulation_grid[:-1])) 
        # self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid 
        # if trace:
        #     print("Translate")
        
    def rotate_clockwise(self):
        """Rotate the grid clockwise when the agent turns left"""
        self.direction = {LEFT: DOWN, DOWN: RIGHT, RIGHT: UP, UP: LEFT}[self.direction]
        # Egocentric simulation
        # self.simulation_grid[:SIM_HIGHT, :SIM_WIDTH] = np.rot90(self.simulation_grid, axes=(1,0))
        # self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid
        # if trace:
        #     print("Rotate clockwise")
    
    def rotate_counterclockwise(self):
        """Rotate the grid counter-clockwise when the agent rotates right"""
        self.direction = {LEFT: UP, DOWN: LEFT, RIGHT: DOWN, UP: RIGHT}[self.direction]
        # Egocentric simulation
        # self.simulation_grid[:SIM_HIGHT, :SIM_WIDTH] = np.rot90(self.simulation_grid)
        # self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid
        # if trace:
        #     print("Rotate counterclockwise")

    def spawn(self, position):
        """Spawn a wall at the position """
        self.simulation_grid[tuple(position)] = WALL
        self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid
        # self.display_grid[tuple(position)] = FEELING_WALL
        
    def kill(self, position):
        """Remove a wall at the position """
        self.simulation_grid[tuple(position)] = EMPTY
        self.display_grid[:SIM_HIGHT, :SIM_WIDTH] = self.simulation_grid
        # self.display_grid[tuple(position)] = FEELING_EMPTY

    def synchronize(self, localized_interactions):
        """Perform the simulator_actions associated with each localized interaction in the list. Return the consistency."""
        self.display_grid[:, :] = self.simulation_grid
        # For each enacted localized interaction
        consistency = True
        for li in localized_interactions:
            # Get the simulator action and position from the action_model et body_model 
            simulator_actions = action_model.get(li.interaction.key(), [])
            ego_position = body_model.get((li.interaction.get_action(), li.position), 0)
            if ego_position == 0:  # Front
                simulator_position = np.mod(self.position + DIRECTIONS[self.direction], SIM_SIZE)
            elif ego_position == 1:  # Left
                simulator_position = np.mod(self.position + DIRECTIONS[(self.direction + 1) % 4], SIM_SIZE)
            else:  # Right
                simulator_position = np.mod(self.position + DIRECTIONS[self.direction - 1], SIM_SIZE)
            # For each simulator action associated with each localized interaction
            for simulator_action in simulator_actions:
                if simulator_action == TRANSLATE:
                    # The previous KILL checked for consistency
                    self.translate()
                elif simulator_action == ROTATE_COUNTERCLOCKWISE:
                    self.rotate_counterclockwise()
                elif simulator_action == ROTATE_CLOCKWISE:
                    self.rotate_clockwise()
                elif simulator_action == KILL:
                    # Check if there was an wall
                    if self.simulation_grid[tuple(simulator_position)] == WALL:
                        consistency = False
                    self.kill(simulator_position)
                    if trace: 
                        print("Kill at", simulator_position)
                elif simulator_action == SPAWN:
                    # Check if it was empty
                    if self.simulation_grid[tuple(simulator_position)] == EMPTY:
                        consistency = False
                    self.spawn(simulator_position)
                    if trace: 
                        print("Spawn at", simulator_position)
        return consistency
    
    def predict(self, action):
        """Simulate the primitive action. Return the outcome."""
        result = [EMPTY]
        self.display_grid[:, :] = self.simulation_grid

        if action == FORWARD:  
            target_position = np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE)
            if self.simulation_grid[tuple(target_position)] == WALL:
                result = [object_model[FORWARD]]
                self.display_grid[tuple(target_position)] = BUMPING
            elif self.simulation_grid[tuple(target_position)] == EMPTY:
                self.translate()
                result = [EMPTY]
            else:
                self.translate()  # Not sure
                result = [UNKNOWN] 
                # self.simulation_grid[:SIM_HIGHT, :SIM_WIDTH] = self.init_grid 
                # self.display_grid[:, :] = self.init_grid 
        
        elif action == TURN_RIGHT:
            self.rotate_counterclockwise()
        
        elif action == TURN_LEFT:
            self.rotate_clockwise()
        
        elif action == FEEL:
            result = []
            for feeling_position in [np.mod(self.position + DIRECTIONS[self.direction], ENV_SIZE), # Front
                                     np.mod(self.position + DIRECTIONS[(self.direction + 1) % 4], ENV_SIZE), # Left 
                                     np.mod(self.position + DIRECTIONS[self.direction - 1], ENV_SIZE)]:  # Right
                if self.simulation_grid[tuple(feeling_position)] == EMPTY:
                    result.append(EMPTY)
                    self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
                elif self.simulation_grid[tuple(feeling_position)] == UNKNOWN:
                    result.append(UNKNOWN)
                    self.display_grid[tuple(feeling_position)] = FEELING_EMPTY
                else:
                    result.append(object_model[FEEL])  # [WALL] 
                    self.display_grid[tuple(feeling_position)] = FEELING_WALL

        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
        
    def clone(self):
        """Return a copy of the simulator in its current state"""
        return Simulator(self.position, self.direction, self.simulation_grid)

    def reset(self):
        """Clear the simulator and move the agent to the origin"""
        self.simulation_grid[:, :] = self.init_grid
        self.display_grid[:, :] = self.init_grid
        self.position = np.array(simulator_position)
        self.direction = UP

# On initialise les modèles de GEITH

* Le `body model` capture les coordonnées égocentriques des événements d'interaction énactés:
* un dictionnaire dont la clé est la décision et le rang de l'outcome, et la valeur sa position.
* Le `action model` capture le mapping entre les enacted interactions et les actions du simulateur:
* dictionnaire dont la clé est l'interaction et la valeur la liste d'action reconnues par le simulateur. 
* Le `object model` permet de predire les outcomes d'une décision en fonction d'un objet présent à une certaine position dans le simulateur:
* Dictionnaire dont la clé est la décition et la valeur un outcome. Cela capture la présence d'un objet mais ne permet pas de distinguer différents objets. 

In [41]:
from itertools import permutations, product

# Le body_model maps (decicion, outcome rank) to spatial position in the simulator
body_model = {(FEEL, 0): 2,  # (-1, 0), # Feel front
              (FEEL, 1): 1,  # (0, -1), # Feel left 
              (FEEL, 2): 0, # (0, 1), # Feel right
              (FORWARD, 0): 0 # (-1, 0) # Bump
}
# action_model maps the primitive interactions to a list of actions to perfom in the simulator
action_model = {
    f"{FEEL}0": [KILL], 
    f"{FEEL}1": [SPAWN], 
    f"{FORWARD}0": [KILL, TRANSLATE],
    f"{FORWARD}1": [SPAWN],
    f"{TURN_LEFT}0": [ROTATE_CLOCKWISE],
    f"{TURN_RIGHT}0": [ROTATE_COUNTERCLOCKWISE]
}
# object_model maps the decisions to the outcome produced by this object (there will be one object_model per type of object)
object_model = {FEEL:WALL,
                FORWARD: WALL}

# Possible permutations
model_permutations = [a + b for a, b in product(permutations([1, 0, 2]), permutations([ROTATE_CLOCKWISE, ROTATE_COUNTERCLOCKWISE]))]
model_permutations = [a + b for a, b in product(permutations([2, 1, 0]), permutations([ROTATE_CLOCKWISE, ROTATE_COUNTERCLOCKWISE]))] #  inverted GEITH
model_permutations

[(2, 1, 0, 4, 3),
 (2, 1, 0, 3, 4),
 (2, 0, 1, 4, 3),
 (2, 0, 1, 3, 4),
 (1, 2, 0, 4, 3),
 (1, 2, 0, 3, 4),
 (1, 0, 2, 4, 3),
 (1, 0, 2, 3, 4),
 (0, 2, 1, 4, 3),
 (0, 2, 1, 3, 4),
 (0, 1, 2, 4, 3),
 (0, 1, 2, 3, 4)]

# Testons l'agent

In [42]:
# Empty grid with sparse objects
grid = [[WALL , WALL , WALL , WALL , WALL ],
        [WALL , EMPTY, EMPTY, EMPTY, WALL ],
        [WALL , EMPTY, WALL , EMPTY, WALL ],
        [WALL , EMPTY, EMPTY, EMPTY, WALL ],
        [WALL , WALL , WALL , WALL , WALL ]]

In [45]:
# Instanciate the environment
e = SmallLoop([1, 1], 0, grid)

# Instanciate the agent 
interactions = [
    Interaction(FORWARD,0,5),
    Interaction(FORWARD,1,-10),
    Interaction(TURN_LEFT,0,-3),
    # Interaction(TURN_LEFT,1,-3),
    Interaction(TURN_RIGHT,0,-3),
    # Interaction(TURN_RIGHT,1,-3),
    Interaction(FEEL,0,-1),
    Interaction(FEEL,1,-1) # ,
    # Interaction(FEEL_LEFT,0,-1),
    # Interaction(FEEL_LEFT,1,-1),
    # Interaction(FEEL_RIGHT,0,-1),
    #Interaction(FEEL_RIGHT,1,-1)
]
# Intialize the simulator to UNKNOWN grid
simulator_grid = np.full((SIM_HIGHT, SIM_WIDTH), UNKNOWN, dtype=int)
simulator_position = np.array([(SIM_WIDTH - 1) // 2, (SIM_HIGHT - 1) // 2])  # Agent at the center of the simulator assuming size is even
simulator = Simulator(simulator_position, UP, simulator_grid)
a = Agent(interactions, simulator)

# Initialize the interaction loop
step = 0
outcome = [0]

# Display
out = Output()
e.display(simulator)
display(out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 640x480 with 1 Axes>', 'i…

In [89]:
step += 1
print("Step:", step)
action = a.action(outcome)
outcome = e.outcome(action)
e.display(simulator)

Step: 1
Enacted primitive localized interactions: [20:5at0],
Sequential predicted outcome: 0, Prediction_correct: True, 
Kill at [1 2]
Intention: 10
Simulate decision: 1 anticipated outcomes: [5, 5, 5] consistent: True


## Run the experiment in a loop 

In [44]:
for step in range(100):
    
    if trace:
        print(f"Step: {step}")
    # step += 1
    action = a.action(outcome)
    outcome = e.outcome(action)
    e.display(simulator)

Step: 0
Enacted primitive localized interactions: [20:5at0],
Sequential predicted outcome: 0, Prediction_correct: True, 
Kill at [1 2]
Intention: 10
Step: 1
Enacted primitive localized interactions: [11:-1at0, 10:-1at1, 11:-1at2],
Sequential predicted outcome: 0, Prediction_correct: False, 
Spawn at [1 3]
Kill at [1 1]
Spawn at [0 2]
Learning (20:5, 11:-1: 1)
Intention: 10
Step: 2
Enacted primitive localized interactions: [11:-1at0, 10:-1at1, 11:-1at2],
Sequential predicted outcome: 0, Prediction_correct: False, 
Spawn at [1 3]
Kill at [1 1]
Spawn at [0 2]
Learning (11:-1, 11:-1: 1)
Learning ((20:5, 11:-1: 1), 11:-1: 1)
Learning (20:5, (11:-1, 11:-1: 1): 1)
Intention: 20
Step: 3
Enacted primitive localized interactions: [21:-10at0],
Sequential predicted outcome: 0, Prediction_correct: False, 
Spawn at [0 2]
Learning (11:-1, 21:-10: 1)
Learning ((11:-1, 11:-1: 1), 21:-10: 1)
Learning (11:-1, (11:-1, 21:-10: 1): 1)
Learning ((20:5, 11:-1: 1), (11:-1, 21:-10: 1): 1)
Intention: 10
Step: 4


## Execute step by step for testing

Note that the simulator is synchronized after the next cycle when the outcome has been recieved. 

# Apprentissage

En présupposant les clés des dictionnaires et la valeurs possibles de chaque clé:
* `body_model` : 3 valeurs possibles x 4 clés = $3^4$
* `action_model` : 5 valeurs possibles x 6 clés = $5^6$
* `object_model` : 2 valeurs possibles x 2 clés = $2^2$

Total 5 062 500 combinaisons

Si on se limite aux permutations sur le `body_model` et le `action_model` des deux decisions turn, ca donne $3 \times 2 \times 2 = 12$

# Questions

* Faut-il stocker des Localized Interaction dans les interactions composites ?
    * Faut il controler la position de l'enaction des interactions quand on incrémente l'interaction composite ?
    * Comment est assignée la position d'une interaction composite ? La position ou est énactée la dernière interaction primitive relative à la position de l'agent au départ de l'interaction composite ? 
* Est-ce qu'il faut créer une intéraction composite qui contient les trois événements d'interaction énactés, ou alors trois intéractions composites ? ou les deux ?
    * Une intéraction composite:
        * Elle match si les trois interactions énactées matchent
    * Trois interactions composites:
        * Elles matchent séparément


# A faire

* Permuter les modèles du geith quand une synchronisation inconsistente est détectée. 
* Il faut que l'agent tende à explorer tant qu'il n'a pas rencontrée d'objet puis reste à priximité quand il en a un jusqu'a ce qu'il est ait calibré ses modèles.
* Il faut qu'il puisse simuler des interactions composites
* Eventuellement vérifier la cohérence d'un modèle avec les séquences d'interaction passées
* Comment générer de nouvelles hypothèses de modèles ?
* Faudrait il que les connaissances liées au différents modèles soient enregistrées sous forme de schémas?
* Il faudra créer les composite interactions à partir d'un ensemble de sous-interactions localisées dans le temps et l'espace
* Pendant l'enaction d'une composite interaction, il faut suivre les déplacements de l'agent pour vérifier que chaque primitive interaction est bien enactée à la position attendue.
* A la création d'une composite interaction, il faut enregistrer sa position a la position de la dernière primitive interaction relative par rapport au début de l'interaciton composite.